# Create Agent with Function Definition

we will create an Agent for Amazon Bedrock using the new capabilities for function definition.

## Prerequisites
Before starting, let's update the botocore and boto3 packages to ensure we have the latest version

In [ ]:
!python3 -m pip install --upgrade -q botocore
!python3 -m pip install --upgrade -q boto3
!python3 -m pip install --upgrade -q awscli

Let's now check the boto3 version to ensure the correct version has been installed. Your version should be greater than or equal to 1.34.90.

In [ ]:
import boto3
import json
import time
import zipfile
from io import BytesIO
import uuid
import pprint
import logging
print(boto3.__version__)

In [ ]:
# setting logger
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

Let's now create the boto3 clients for the required AWS services

In [ ]:
# getting boto3 clients for required AWS services
sts_client = boto3.client('sts')
iam_client = boto3.client('iam')
lambda_client = boto3.client('lambda')
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime')

Next we can set some configuration variables for the agent and for the lambda function being created

In [ ]:
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]
region, account_id

In [ ]:
# configuration variables
suffix = f"{region}-{account_id}"
agent_name = "ai-shopping-assistant"
agent_bedrock_allow_policy_name = f"{agent_name}-ba-{suffix}"
agent_role_name = f'AmazonBedrockExecutionRoleForAgents_{agent_name}'
agent_foundation_model = "anthropic.claude-3-sonnet-20240229-v1:0"
agent_description = "Agent for shopping assistance to users"
agent_instruction = "You are a shopping assistant, helping shoppers to pick right products based on their needs referring to the product catalog. The product catalog contains only products related footwear and clothing categories. When the user asks for products which do not fall under the categories, footwear or clothing, insist them gently to ask for products in these two categories only. For example, Backpacks and bags do not belong to footwear or clothing category. When you receive a query that has multiple questions or products in the same query, break it into multiple queries and run one by one using the right tools. Always generate some additional relevant follow up questions inside <question></question>. When you generate a product image, always ask the user if they want to retrieve similar relevant items from the product catalog based on the text description and generated image."
agent_action_group_name = "ShoppingActionGroup"
agent_action_group_description = "Actions for getting the question from the user, perform actions only based on the provided tools. Always, depend on the tool description to decide which tool to use for a question."
agent_alias_name = f"{agent_name}-alias"
lambda_function_name = "shopping-assistant-function-"+account_id 


## Create Agent Role

We will now create the agent. To do so, we first need to create the agent policies that allow bedrock model invocation for a specific foundation model and the agent IAM role with the policy associated to it. 

In [ ]:
# Create IAM policies for agent
bedrock_agent_bedrock_allow_policy_statement = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AmazonBedrockAgentBedrockFoundationModelPolicy",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": [
                "arn:aws:bedrock:"+region+":"+account_id+":foundation-model/anthropic.claude-3-sonnet-20240229-v1:0"
            ]
        }
    ]
}

bedrock_policy_json = json.dumps(bedrock_agent_bedrock_allow_policy_statement)

agent_bedrock_policy = iam_client.create_policy(
    PolicyName=agent_bedrock_allow_policy_name,
    PolicyDocument=bedrock_policy_json
)



In [ ]:
# Create IAM Role for the agent and attach IAM policies
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
          "Effect": "Allow",
          "Principal": {
            "Service": "bedrock.amazonaws.com"
          },
          "Action": "sts:AssumeRole"
    }]
}

assume_role_policy_document_json = json.dumps(assume_role_policy_document)
agent_role = iam_client.create_role(
    RoleName=agent_role_name,
    AssumeRolePolicyDocument=assume_role_policy_document_json
)

# Pause to make sure role is created
time.sleep(10)
    
iam_client.attach_role_policy(
    RoleName=agent_role_name,
    PolicyArn=agent_bedrock_policy['Policy']['Arn']
)

### Creating the agent
Once the needed IAM role is created, we can use the Bedrock Agent client to create a new agent. To do so we use the `create_agent` function. It requires an agent name, underlying foundation model and instructions. You can also provide an agent description. Note that the agent created is not yet prepared. Later, we will prepare and use the agent.

In [ ]:
response = bedrock_agent_client.create_agent(
    agentName=agent_name,
    agentResourceRoleArn=agent_role['Role']['Arn'],
    description=agent_description,
    idleSessionTTLInSeconds=1800,
    foundationModel=agent_foundation_model,
    instruction=agent_instruction,
)
agent_id = response['agent']['agentId']
agent_id

Let's now store the agent id in a local variable to use it on subsequent steps.

## Create Agent Action Group
We will now create an agent action group that uses the lambda function created earlier. The [`create_agent_action_group`](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent/client/create_agent_action_group.html) function provides this functionality. We will use `DRAFT` as the agent version since we haven't yet created an agent version or alias. To inform the agent about the action group capabilities, we provide an action group description.

In this example, we provide the Action Group functionality using a `functionSchema`. You can alternatively provide an `APISchema`. The notebook [02-create-agent-with-api-schema.ipynb](02-create-agent-with-api-schema/02-create-agent-with-api-schema.ipynb) provides an example of that approach.

To define the functions using a function schema, you need to provide the `name`, `description` and `parameters` for each function.

In [ ]:
agent_functions = [
    {
        'name': 'get_relevant_items_for_image',
        'description': "This tool gets the relevant products from product catalog based on user's query. Use this tool only you want to retrieve items from the product catalog and only when the query contains an image in addition to the text description. Use this tool when you want to retrieve items similar to a product image. Use the text_description used to generate the image as the text query. Don't use this function when the user provides only a text query as input. When the user's query is not specific and a general recommendation related query, then use get_any_general_recommendation tool. When you see the tool's results are totally irrelevant to the user query, then try with other appropriate tools. Don't do a stringent test to check if the products are exactly relevant. ",
        'parameters': {
            "s3_image_url": {
                "description": "s3 location or web url of the image which was used to retrieve relevant items ",
                "required": True,
                "type": "string"
            },
             "text_description": {
                "description": "User's text description that was used to generate image",
                "required": False,
                "type": "string"
            }
        }
    },
    {
        'name': 'generate_images',
        'description': "Generate AI images of product based on text description. Use this function only to generate or create an image for a product the user have in their mind. Make sure to adjust the prompt so that it is not violating the AWS Responsible AI Policy. Always ask the user if they want to retrieve similar relevant items from the product catalog based on the text description and generated image.",
        'parameters': {
            "text_description": {
                "description": "User's text description describing the product",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'get_more_product_details',
        'description': "This tool gives answer for the customer's question on the specific product, related to it's attributes stored in the catalog. To provide any recommendations based on a product, use the tool, get_any_general_recommendation with product_id and product image inputs.",
        'parameters': {
            "product_id": {
                "description": "unique identifier of the show product to get product details",
                "required": True,
                "type": "string"
            },
            "question": {
                "description": "Customer's question on the product",
                "required": True,
                "type": "string"
            }
        }
    },
       {
        'name': 'get_relevant_items_for_text',
        'description': "This tool gets the relevant items from product catalog based on user's query. Use this function only when the user wants to retrieve items from the product catalog and only when the user query is a text based query. Don't use this function when the user provides an image as query input. When the user's query is not specific and a general recommendation related query, then use get_any_general_recommendation tool. Only when you see the tool's results are totally irrelevant to the user query, then try with other appropriate tools. Don't do a stringent test to check if the products are exactly relevant. ",
        'parameters': {
            "text_query": {
                "description": "text query input from the user to retrieve products",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'get_more_product_details_from_internet',
        'description': "Use this function to answer questions on a product by looking at a webpage that you open through the given URL. Use this function only when you are not able to answer the user's question through 'get_more_product_details' tool.",
        'parameters': {
            "product_id": {
                "description": "unique identifier of the products",
                "required": True,
                "type": "string"
            },
            "question": {
                "description": "Customer's question on the product",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'retrieve_with_keyword_search',
        'description': "This tool gets the items from product catalog based on user's query by matching exact keywords between query and product information. Use this function only when the query contains entity names where exact matching is necessary. These entities can include brand names, person names, abbreviations etc. You can think an entity as any term that should be called exactly as it is and giving similar terms to them makes no sense. Examples - Nike, Adidas, iOS etc. When you don't see the tool's results relevant to the user query, then try with other relevant tools. Do not use this tool for getting any general recommendation related queries.",
        'parameters': {
            "text_query": {
                "description": "text query input from the user to retrieve products or User's text description that was used to generate image",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'get_any_general_recommendation',
        'description': "This tool can be used to provide crisp fashion related recommendations related to a specific product or in general. Always, generate a relevant image using generate_images tool to visually present your recommendation.",
        'parameters': {
            "product_id": {
                "description": "unique identifier of the show product to get product details",
                "required": False,
                "type": "string"
            },
             "question": {
                "description": "User's question",
                "required": True,
                "type": "string"
            },
             "s3_image_url": {
                "description": "s3 location or web url of the product image",
                "required": False,
                "type": "string"
            }
        }
    }
    
]

In [ ]:
# update lambda environment variables

response = client.update_function_configuration(
    FunctionName=lambda_function_name,
   
    Environment={
        'Variables': {
            'DOMAIN_ENDPOINT': 'string',
            'REGION': region,
            'TEXT_EMBEDDING_MODEL': 'string',
            'IMAGE_EMBEDDING_MODEL': 'string',
            'INDEX_NAME': 'string',
            'S3_BUCKET':account_id+"-ml-search"
        }
    }
)

# DOMAIN_ENDPOINT = os.environ['DOMAIN_ENDPOINT']#'search-opensearchservi-vjfctcacr9us-pqhlanotok3kipumbztwmtgfaq.us-east-1.es.amazonaws.com'
# REGION = json.loads(os.environ['REGION'])#REGION 
# TEXT_EMBEDDING_MODEL = json.loads(os.environ['TEXT_EMBEDDING_MODEL'])
# IMAGE_EMBEDDING_MODEL = json.loads(os.environ['IMAGE_EMBEDDING_MODEL'])
# INDEX_NAME = json.loads(os.environ['INDEX_NAME'])#"esci-us-clothing"
# S3_BUCKET = json.loads(os.environ['S3'])#bedrock-video-generation-us-east-1-lbxkrh

In [ ]:
# Pause to make sure agent is created
#time.sleep(30)
# Now, we can configure and create an action group here:
agent_action_group_response = bedrock_agent_client.create_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupExecutor={
        'lambda': "arn:aws:lambda:"+region+":"+account_id+":function:shopping-assistant-function-"+account_id
    },
    actionGroupName=agent_action_group_name,
    functionSchema={
        'functions': agent_functions
    },
    description=agent_action_group_description
)
agent_action_group_response

## Allowing Agent to invoke Action Group Lambda
Before using the action group, we need to allow the agent to invoke the lambda function associated with the action group. This is done via resource-based policy. Let's add the resource-based policy to the lambda function created

In [ ]:
# Create allow invoke permission on lambda
response = lambda_client.add_permission(
    FunctionName=lambda_function_name,
    StatementId='allow_bedrock',
    Action='lambda:InvokeFunction',
    Principal='bedrock.amazonaws.com',
    SourceArn=f"arn:aws:bedrock:{region}:{account_id}:agent/{agent_id}",
)
response

## Preparing Agent

Let's create a DRAFT version of the agent that can be used for internal testing.


In [ ]:
response = bedrock_agent_client.prepare_agent(
    agentId=agent_id
)
print(response)

In [ ]:
# Pause to make sure agent is prepared
#time.sleep(30)

response = bedrock_agent_client.create_agent_alias(
    agentId=response['agentId'],
    agentAliasName = "prod"
    )

# Extract the agentAliasId from the response
agent_alias_id = response['agentAlias']['agentAliasId']

## Invoke Agent

Now that we've created the agent, let's use the `bedrock-agent-runtime` client to invoke this agent and perform some tasks.

In [ ]:
## create a random id for session initiator id
session_id:str = str(uuid.uuid1())
enable_trace:bool = False
end_session:bool = False

# invoke the agent API
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="Give me some black shoes for boys",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

In [ ]:
%%time
event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
            # End event indicates that the request finished successfully
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

In [ ]:
# And here is the response if you just want to see agent's reply
print(agent_answer)

In [ ]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="great. please reserve one day of time off, June 1 2024",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

In [ ]:
%%time
event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
            # End event indicates that the request finished successfully
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

In [ ]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="now let me take the last 3 months of the year off as vacation, from Oct 1 2024 through Dec 31 2024",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

In [ ]:
%%time
event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
            # End event indicates that the request finished successfully
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

In [ ]:
def simple_agent_invoke(input_text, agent_id, agent_alias_id, session_id=None, enable_trace=False, end_session=False):
    agentResponse = bedrock_agent_runtime_client.invoke_agent(
        inputText=input_text,
        agentId=agent_id,
        agentAliasId=agent_alias_id, 
        sessionId=session_id,
        enableTrace=enable_trace, 
        endSession= end_session
    )
    logger.info(pprint.pprint(agentResponse))
    
    event_stream = agentResponse['completion']
    try:
        for event in event_stream:        
            if 'chunk' in event:
                data = event['chunk']['bytes']
                logger.info(f"Final answer ->\n{data.decode('utf8')}")
                agent_answer = data.decode('utf8')
                end_event_received = True
                # End event indicates that the request finished successfully
            elif 'trace' in event:
                logger.info(json.dumps(event['trace'], indent=2))
            else:
                raise Exception("unexpected event.", event)
    except Exception as e:
        raise Exception("unexpected event.", e)

In [ ]:
simple_agent_invoke("how much time off does employee 2 have?", agent_id, agent_alias_id, session_id)

In [ ]:
simple_agent_invoke("reserve July 30 2024 through August 4 2024 please", agent_id, agent_alias_id, session_id)

In [ ]:
simple_agent_invoke("how many days does employee 9 have?", agent_id, agent_alias_id, session_id, enable_trace=True)

## Clean up (optional)

The next steps are optional and demonstrate how to delete our agent. To delete the agent we need to:

1. update the action group to disable it
2. delete agent action group
4. delete agent


In [ ]:
# This is not needed, you can delete agent successfully after deleting alias only
# Additionaly, you need to disable it first
action_group_id = agent_action_group_response['agentActionGroup']['actionGroupId']
action_group_name = agent_action_group_response['agentActionGroup']['actionGroupName']

response = bedrock_agent_client.update_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupId= action_group_id,
    actionGroupName=action_group_name,
    actionGroupExecutor=actionGroupExecutor={
        'lambda': "arn:aws:lambda:"+region+":"+account_id+":function:shopping-assistant-function-"+account_id
    },
    functionSchema={
        'functions': agent_functions
    },
    actionGroupState='DISABLED',
)

action_group_deletion = bedrock_agent_client.delete_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupId= action_group_id
)

In [ ]:
agent_deletion = bedrock_agent_client.delete_agent(
    agentId=agent_id
)

## Conclusion
We have now experimented with using boto3 SDK to create, invoke and delete an agent created using function definitions.

## Take aways
Adapt this notebook to create new agents using function definitions for your application

## Thank You